In [1]:
# from dotenv import load_dotenv
# load_dotenv()

In [2]:
import json
import textwrap
import pandas as pd
from ipywidgets import widgets
from IPython.display import display
from ipywidgets import HBox
import matplotlib.pyplot as plt
from tabulate import tabulate

In [3]:
# Load the wandb table data
def load_wandb_data(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
    return pd.DataFrame(data['data'], columns=data['columns'])

In [4]:
def fixedwidth(text):
    """Wrap text to fixed width"""
    return "\n".join(textwrap.wrap(str(text), width=160, replace_whitespace=False))

def format_row(row):
    """Format a single row for display as a table"""
    data = [
        ["Step", row['Step']],
        ["Reward", f"{row['Reward']:.2f}"],
        ["Input", row['Input']],
        ["Prompt", fixedwidth(row['Prompt'])], 
        ["Completion", fixedwidth(row['Completion'])],
    ]
    return tabulate(data, tablefmt='grid')

def present_row(row):
    """Display a formatted row as table"""
    print(format_row(row))

In [5]:
def create_browse_app(df):
    """Create an interactive widget to browse through the data"""
    def browse_data(i=0):
        row = df.iloc[i]
        present_row(row)

    index = widgets.IntText(value=0, description='Index:')
    left_button = widgets.Button(description='Previous')
    right_button = widgets.Button(description='Next')

    def on_left_button_clicked(b):
        if index.value > 0:
            index.value -= 1

    def on_right_button_clicked(b):
        if index.value < len(df) - 1:
            index.value += 1

    left_button.on_click(on_left_button_clicked)
    right_button.on_click(on_right_button_clicked)

    ui = HBox([left_button, index, right_button])
    out = widgets.interactive_output(browse_data, {'i': index})

    display(ui, out)

In [6]:
def preprocess_df(df):
    df['Step'] = df['Step'].astype(int)
    df['Reward'] = df['Reward'].astype(float)
    df['Input'] = df['Input'].astype(str)
    df['Prompt'] = df['Prompt'].astype(str)
    df['Completion'] = df['Completion'].astype(str)
    df['Prompt'] = df['Prompt'].map(lambda x: x.rsplit('**user**:', 1)[-1])
    df.sort_values(by="Step", ascending=False, inplace=True)
    return df

In [ ]:
# Load and display your data
from pathlib import Path

# read all json files in the directory using pathlib
run_name = 'run-20250413_235627-w1ck91gt'
directory = f"../wandb/{run_name}/files/media/table"
files = list(Path(directory).glob("interactions_*.json"))

# load all json files
df = pd.concat([load_wandb_data(f) for f in files])
df = preprocess_df(df)
print(len(df))

In [ ]:
df.loc[df['Completion'].str.contains("timed out")]

In [ ]:
# Create the interactive browser
create_browse_app(df)

In [ ]:
retrieval_attempt_mask = df['Completion'].str.contains("timed out")
df.loc[retrieval_attempt_mask]

In [ ]:
# plot average reward by step
# Plot raw average and moving average
means = df.groupby('Step')['Reward'].mean()
means.plot(alpha=0.5, label='Raw Average')
means.rolling(window=5).mean().plot(label='5-step Moving Average')
plt.legend()

In [12]:
from ast import literal_eval

def extract_titles(completion):
    # Extract titles from the completion
    for line in completion.splitlines():
        if line.startswith('#'):
            yield line.split('#')[1].strip()

# Apply the function to the 'Completion' column
df['Retrieved Titles'] = df['Completion'].apply(lambda x: list(extract_titles(x)))

df['Supporting Titles'] = df['Input'].map(lambda x: x.split('supporting_titles:', 1)[-1]).map(literal_eval)


In [ ]:
df['Retrieved Titles'].map(len).hist()

In [14]:
df['Successfully Retrieved Titles'] = df.apply(lambda row: set(row['Retrieved Titles']).intersection(row['Supporting Titles']), axis=1)
df['retrieval.recall'] = df['Successfully Retrieved Titles'].map(len) / df['Supporting Titles'].map(len)
df['retrieval.precision'] = df['Successfully Retrieved Titles'].map(len) / df['Retrieved Titles'].map(len)

In [ ]:
retrieval_attempt_mask = df['Retrieved Titles'].map(len) > 0
len(df[retrieval_attempt_mask]) / len(df)

In [ ]:
df.loc[retrieval_attempt_mask, 'retrieval.recall'].describe()

In [ ]:
df.loc[retrieval_attempt_mask, 'retrieval.precision'].describe()

In [ ]:
error_mask = df['Completion'].str.contains("Error")
len(df.loc[error_mask]) / len(df)


In [ ]:
df['Reward'].hist()

In [20]:
# plot reward distribution characteristics per prompt
prompt_reward_stats = df.groupby('Prompt')['Reward'].describe()

In [ ]:
prompt_reward_stats.plot.scatter(x='std', y='mean', figsize=(10, 10))